In [42]:
!pip install -q faker scikit-learn pandas numpy joblib networkx plotly

In [2]:
#2
import numpy as np
import pandas as pd
from faker import Faker
from pathlib import Path

fake = Faker()

np.random.seed(42)

print("RiskGraph AI environment ready.")

RiskGraph AI environment ready.


In [3]:
#3
import numpy as np
import pandas as pd

np.random.seed(42)

# ============================================================
# CONFIGURATION
# ============================================================

N_TRANSACTIONS = 20000
N_CUSTOMERS = 4000
N_MERCHANTS = 300
N_DEVICES = 5000
N_IPS = 6000

customers = [
    f"CUST_{i:05d}"
    for i in range(N_CUSTOMERS)
]

merchants = [
    f"MER_{i:04d}"
    for i in range(N_MERCHANTS)
]

devices = [
    f"DEV_{i:05d}"
    for i in range(N_DEVICES)
]

ips = [
    f"IP_{i:05d}"
    for i in range(N_IPS)
]

locations = [
    "Mumbai",
    "Delhi",
    "Bengaluru",
    "Hyderabad",
    "Chennai",
    "Pune",
    "Kolkata",
    "Ahmedabad",
    "Jaipur",
    "Kochi"
]


# ============================================================
# CUSTOMER PROFILES
# ============================================================

customer_profiles = {}

for customer in customers:

    customer_profiles[customer] = {

        "account_age_days": np.random.randint(
            30, 1500
        ),

        "normal_amount": np.random.lognormal(
            mean=np.log(1200),
            sigma=0.7
        ),

        "home_location": np.random.choice(
            locations
        )
    }


# ============================================================
# TRANSACTION GENERATION
# ============================================================

rows = []

start_time = pd.Timestamp(
    "2026-01-01"
)


for i in range(N_TRANSACTIONS):

    customer = np.random.choice(customers)

    profile = customer_profiles[customer]

    merchant = np.random.choice(merchants)

    device = np.random.choice(devices)

    ip = np.random.choice(ips)


    timestamp = (
        start_time
        + pd.Timedelta(
            minutes=int(
                np.random.randint(
                    0,
                    180 * 24 * 60
                )
            )
        )
    )


    # Normal transaction amount

    amount = np.random.lognormal(
        mean=np.log(
            profile["normal_amount"]
        ),
        sigma=0.45
    )

    amount = round(
        float(
            np.clip(
                amount,
                50,
                100000
            )
        ),
        2
    )


    # Behavioral features

    device_age_days = np.random.randint(
        1,
        1000
    )

    transactions_last_10min = np.random.poisson(
        1.2
    )

    failed_attempts = np.random.poisson(
        0.5
    )


    location_change = (
        np.random.random() < 0.05
    )


    location = profile[
        "home_location"
    ]


    if location_change:

        other_locations = [
            x for x in locations
            if x != profile["home_location"]
        ]

        location = np.random.choice(
            other_locations
        )


    # Amount deviation

    amount_deviation = (
        amount /
        max(
            profile["normal_amount"],
            1
        )
    )


    # ========================================================
    # FRAUD PROBABILITY
    # ========================================================

    fraud_probability = 0.015


    # New account

    if profile[
        "account_age_days"
    ] < 60:

        fraud_probability += 0.08


    # Large amount deviation

    if amount_deviation > 5:

        fraud_probability += 0.18

    elif amount_deviation > 3:

        fraud_probability += 0.08


    # High velocity

    if transactions_last_10min >= 5:

        fraud_probability += 0.20

    elif transactions_last_10min >= 3:

        fraud_probability += 0.08


    # Failed attempts

    if failed_attempts >= 4:

        fraud_probability += 0.15

    elif failed_attempts >= 2:

        fraud_probability += 0.05


    # Location anomaly

    if location_change:

        fraud_probability += 0.10


    # New device

    if device_age_days < 7:

        fraud_probability += 0.12

    elif device_age_days < 30:

        fraud_probability += 0.04


    # ========================================================
    # COMBINED RISK PATTERN
    # ========================================================

    combined_signals = 0


    if amount_deviation > 4:
        combined_signals += 1

    if transactions_last_10min >= 4:
        combined_signals += 1

    if failed_attempts >= 3:
        combined_signals += 1

    if location_change:
        combined_signals += 1

    if device_age_days < 14:
        combined_signals += 1


    if combined_signals >= 3:

        fraud_probability += 0.25


    fraud_probability = min(
        fraud_probability,
        0.95
    )


    # Fraud label

    is_fraud = int(
        np.random.random()
        < fraud_probability
    )


    rows.append({

        "transaction_id":
            f"TX_{i:07d}",

        "customer_id":
            customer,

        "merchant_id":
            merchant,

        "device_id":
            device,

        "ip_id":
            ip,

        "timestamp":
            timestamp,

        "amount":
            amount,

        "location":
            location,

        "account_age_days":
            profile[
                "account_age_days"
            ],

        "device_age_days":
            device_age_days,

        "transactions_last_10min":
            transactions_last_10min,

        "failed_attempts":
            failed_attempts,

        "location_change":
            int(location_change),

        "amount_deviation":
            round(
                amount_deviation,
                3
            ),

        "is_fraud":
            is_fraud
    })


# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(rows)

df = df.sort_values(
    "timestamp"
).reset_index(drop=True)


print("=" * 60)
print("RISKGRAPH AI DATASET")
print("=" * 60)

print(
    f"Transactions : {len(df):,}"
)

print(
    f"Fraud cases  : {df['is_fraud'].sum():,}"
)

print(
    f"Fraud rate   : "
    f"{df['is_fraud'].mean() * 100:.2f}%"
)

print("=" * 60)

df.head()

RISKGRAPH AI DATASET
Transactions : 20,000
Fraud cases  : 762
Fraud rate   : 3.81%


,transaction_id,customer_id,merchant_id,device_id,ip_id,timestamp,amount,location,account_age_days,device_age_days,transactions_last_10min,failed_attempts,location_change,amount_deviation,is_fraud
0,TX_0018519,CUST_03141,MER_0241,DEV_03669,IP_02184,2026-01-01 00:09:00,3825.04,Kolkata,812,940,0,0,0,0.883,0
1,TX_0003521,CUST_01455,MER_0107,DEV_02829,IP_01908,2026-01-01 00:27:00,470.24,Mumbai,1116,757,0,1,0,0.598,0
2,TX_0010657,CUST_02329,MER_0148,DEV_00527,IP_00409,2026-01-01 00:37:00,182.74,Kochi,1460,414,0,0,0,0.537,0
3,TX_0006411,CUST_01288,MER_0236,DEV_01351,IP_04566,2026-01-01 00:48:00,1068.51,Kolkata,1146,138,1,1,0,1.252,0
4,TX_0000307,CUST_03362,MER_0107,DEV_03479,IP_03020,2026-01-01 00:51:00,608.43,Ahmedabad,597,112,0,1,0,1.218,0


In [4]:
#4
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFraud distribution:")
print(df["is_fraud"].value_counts())

print("\nFraud percentage:")
print(
    df["is_fraud"]
    .value_counts(normalize=True)
    * 100
)

(20000, 15)

Columns:
['transaction_id', 'customer_id', 'merchant_id', 'device_id', 'ip_id', 'timestamp', 'amount', 'location', 'account_age_days', 'device_age_days', 'transactions_last_10min', 'failed_attempts', 'location_change', 'amount_deviation', 'is_fraud']

Fraud distribution:
is_fraud
0    19238
1      762
Name: count, dtype: int64

Fraud percentage:
is_fraud
0    96.19
1     3.81
Name: proportion, dtype: float64


In [5]:
#5
df.to_csv(
    "transactions.csv",
    index=False
)

print("transactions.csv created successfully.")

transactions.csv created successfully.


In [7]:
#phase - 2
#Cell 6 — Prepare the ML data
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import numpy as np
import pandas as pd

In [8]:
#7
# Work on a copy
model_df = df.copy()

# Convert timestamp
model_df["timestamp"] = pd.to_datetime(
    model_df["timestamp"]
)

# Time features
model_df["hour"] = (
    model_df["timestamp"].dt.hour
)

model_df["day_of_week"] = (
    model_df["timestamp"].dt.dayofweek
)

model_df["is_weekend"] = (
    model_df["day_of_week"] >= 5
).astype(int)


# Amount features
model_df["log_amount"] = np.log1p(
    model_df["amount"]
)

model_df["high_value_transaction"] = (
    model_df["amount"] > 10000
).astype(int)


# Behavioral indicators
model_df["high_velocity"] = (
    model_df["transactions_last_10min"] >= 4
).astype(int)

model_df["high_failure_activity"] = (
    model_df["failed_attempts"] >= 3
).astype(int)

model_df["new_device"] = (
    model_df["device_age_days"] < 14
).astype(int)

model_df["new_account"] = (
    model_df["account_age_days"] < 60
).astype(int)


# Combined behavioral signal
model_df["behavior_risk_count"] = (

    model_df["high_velocity"]

    + model_df["high_failure_activity"]

    + model_df["new_device"]

    + model_df["new_account"]

    + model_df["location_change"]

    + (
        model_df["amount_deviation"] > 3
    ).astype(int)
)


print(
    model_df[
        [
            "amount",
            "amount_deviation",
            "high_velocity",
            "new_device",
            "behavior_risk_count",
            "is_fraud"
        ]
    ].head()
)

    amount  amount_deviation  high_velocity  new_device  behavior_risk_count  \
0  3825.04             0.883              0           0                    0   
1   470.24             0.598              0           0                    0   
2   182.74             0.537              0           0                    0   
3  1068.51             1.252              0           0                    0   
4   608.43             1.218              0           0                    0   

   is_fraud  
0         0  
1         0  
2         0  
3         0  
4         0  


In [9]:
#8
features = [

    "amount",

    "log_amount",

    "account_age_days",

    "device_age_days",

    "transactions_last_10min",

    "failed_attempts",

    "location_change",

    "amount_deviation",

    "hour",

    "day_of_week",

    "is_weekend",

    "high_value_transaction",

    "high_velocity",

    "high_failure_activity",

    "new_device",

    "new_account",

    "behavior_risk_count"
]


X = model_df[features]

y = model_df["is_fraud"]


print("Feature matrix:", X.shape)

print("Target:", y.shape)

Feature matrix: (20000, 17)
Target: (20000,)


In [10]:
#9
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)


print(
    "Training samples:",
    len(X_train)
)

print(
    "Held-out test samples:",
    len(X_test)
)

print(
    "\nTraining fraud rate:",
    f"{y_train.mean() * 100:.2f}%"
)

print(
    "Test fraud rate:",
    f"{y_test.mean() * 100:.2f}%"
)

Training samples: 16000
Held-out test samples: 4000

Training fraud rate: 3.81%
Test fraud rate: 3.80%


In [11]:
#10
model = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    min_samples_leaf=4,

    class_weight="balanced",

    random_state=42,

    n_jobs=-1
)


print("Training model...")

model.fit(
    X_train,
    y_train
)

print("Model training complete.")

Training model...
Model training complete.


In [12]:
#11
# Probability of fraud
y_probability = model.predict_proba(
    X_test
)[:, 1]


# Classification threshold
threshold = 0.50

y_pred = (
    y_probability >= threshold
).astype(int)


print("Predictions generated.")

print(
    "\nFirst 10 fraud probabilities:"
)

print(
    y_probability[:10]
)

Predictions generated.

First 10 fraud probabilities:
[0.19843243 0.17333196 0.1954255  0.167566   0.10062189 0.18860213
 0.17858541 0.1917041  0.18423041 0.46499177]


In [13]:
#12
precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

cm = confusion_matrix(
    y_test,
    y_pred
)


print("=" * 60)
print("RISKGRAPH AI — HELD-OUT TEST RESULTS")
print("=" * 60)

print(
    f"\nPrecision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)

print(
    f"F1 Score  : {f1:.4f}"
)

print(
    f"ROC-AUC   : {roc_auc:.4f}"
)

print("\nConfusion Matrix:")

print(cm)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Legitimate",
            "Fraud"
        ],
        zero_division=0
    )
)

RISKGRAPH AI — HELD-OUT TEST RESULTS

Precision : 0.0879
Recall    : 0.1776
F1 Score  : 0.1176
ROC-AUC   : 0.6671

Confusion Matrix:
[[3568  280]
 [ 125   27]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.97      0.93      0.95      3848
       Fraud       0.09      0.18      0.12       152

    accuracy                           0.90      4000
   macro avg       0.53      0.55      0.53      4000
weighted avg       0.93      0.90      0.91      4000



In [14]:
#13
importance = pd.DataFrame({

    "feature": features,

    "importance":
        model.feature_importances_
})

importance = (
    importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    importance.head(10)
)

                   feature  importance
0  transactions_last_10min    0.178851
1      behavior_risk_count    0.130288
2         amount_deviation    0.095397
3          device_age_days    0.094656
4         account_age_days    0.090922
5                   amount    0.081994
6               log_amount    0.080942
7          failed_attempts    0.065748
8                     hour    0.058988
9          location_change    0.051178


In [15]:
#14
import joblib

joblib.dump(
    {
        "model": model,
        "features": features
    },
    "fraud_model.joblib"
)

print(
    "Model saved as fraud_model.joblib"
)

Model saved as fraud_model.joblib


In [16]:
#15
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

thresholds = np.arange(0.05, 0.96, 0.05)

for threshold in thresholds:

    predictions = (
        y_probability >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4)
    })


threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df

,threshold,precision,recall,f1
0,0.05,0.0380,1.0000,0.0732
1,0.10,0.0383,0.9868,0.0738
2,0.15,0.0442,0.8158,0.0838
3,0.20,0.0645,0.6382,0.1171
4,0.25,0.0838,0.5526,0.1456
5,0.30,0.0938,0.5197,0.1590
6,0.35,0.0940,0.4408,0.1549
7,0.40,0.0950,0.3618,0.1505
8,0.45,0.0909,0.2697,0.1360
9,0.50,0.0879,0.1776,0.1176


In [17]:
#16
importance.head(15)

,feature,importance
0,transactions_last_10min,0.178851
1,behavior_risk_count,0.130288
2,amount_deviation,0.095397
3,device_age_days,0.094656
4,account_age_days,0.090922
5,amount,0.081994
6,log_amount,0.080942
7,failed_attempts,0.065748
8,hour,0.058988
9,location_change,0.051178


In [18]:
#17
print(
    model_df.groupby("is_fraud")[
        [
            "amount_deviation",
            "transactions_last_10min",
            "failed_attempts",
            "device_age_days",
            "location_change",
            "account_age_days"
        ]
    ].mean()
)

          amount_deviation  transactions_last_10min  failed_attempts  \
is_fraud                                                               
0                 1.101185                 1.179177         0.493139   
1                 1.136839                 1.923885         0.700787   

          device_age_days  location_change  account_age_days  
is_fraud                                                      
0              500.334546         0.045587        778.011280  
1              483.288714         0.177165        753.129921  


In [21]:
#18
import numpy as np
import pandas as pd

np.random.seed(42)

# ============================================================
# RISKGRAPH AI — SYNTHETIC DATASET V2
# ============================================================

N_TRANSACTIONS = 20000
N_CUSTOMERS = 4000
N_MERCHANTS = 300
N_DEVICES = 5000
N_IPS = 6000

customers = [
    f"CUST_{i:05d}"
    for i in range(N_CUSTOMERS)
]

merchants = [
    f"MER_{i:04d}"
    for i in range(N_MERCHANTS)
]

devices = [
    f"DEV_{i:05d}"
    for i in range(N_DEVICES)
]

ips = [
    f"IP_{i:05d}"
    for i in range(N_IPS)
]

locations = [
    "Mumbai",
    "Delhi",
    "Bengaluru",
    "Hyderabad",
    "Chennai",
    "Pune",
    "Kolkata",
    "Ahmedabad",
    "Jaipur",
    "Kochi"
]


# ============================================================
# CUSTOMER BEHAVIOR PROFILES
# ============================================================

customer_profiles = {}

for customer in customers:

    customer_profiles[customer] = {

        "account_age_days": int(
            np.random.gamma(
                shape=4,
                scale=220
            )
        ) + 30,

        "normal_amount": float(
            np.random.lognormal(
                mean=np.log(1200),
                sigma=0.55
            )
        ),

        "home_location": np.random.choice(
            locations
        ),

        "primary_device": np.random.choice(
            devices
        ),

        "primary_ip": np.random.choice(
            ips
        )
    }


# ============================================================
# TRANSACTION GENERATION
# ============================================================

rows = []

start_time = pd.Timestamp("2026-01-01")

fraud_count = 0


for i in range(N_TRANSACTIONS):

    customer = np.random.choice(customers)

    profile = customer_profiles[customer]

    # --------------------------------------------------------
    # Decide whether this transaction belongs to a fraud
    # scenario.
    #
    # Target roughly 5-7% fraud.
    # --------------------------------------------------------

    is_fraud = int(
        np.random.random() < 0.06
    )

    if is_fraud:
        fraud_count += 1


    # --------------------------------------------------------
    # Transaction timestamp
    # --------------------------------------------------------

    timestamp = (
        start_time
        + pd.Timedelta(
            minutes=int(
                np.random.randint(
                    0,
                    180 * 24 * 60
                )
            )
        )
    )


    # --------------------------------------------------------
    # Normal transaction behavior
    # --------------------------------------------------------

    amount = np.random.lognormal(
        mean=np.log(
            profile["normal_amount"]
        ),
        sigma=0.40
    )

    device = profile["primary_device"]

    ip = profile["primary_ip"]

    location = profile["home_location"]

    device_age_days = np.random.randint(
        30,
        1200
    )

    transactions_last_10min = np.random.poisson(
        1.1
    )

    failed_attempts = np.random.poisson(
        0.4
    )

    location_change = 0


    # ========================================================
    # FRAUD SCENARIOS
    # ========================================================

    if is_fraud:

        scenario = np.random.choice(
            [
                "velocity",
                "amount",
                "new_device",
                "location",
                "credential_abuse",
                "combined"
            ],
            p=[
                0.18,
                0.15,
                0.15,
                0.12,
                0.15,
                0.25
            ]
        )


        # ----------------------------------------------------
        # Scenario 1: High velocity
        # ----------------------------------------------------

        if scenario == "velocity":

            transactions_last_10min = np.random.randint(
                5,
                15
            )

            failed_attempts = np.random.randint(
                1,
                5
            )


        # ----------------------------------------------------
        # Scenario 2: Amount anomaly
        # ----------------------------------------------------

        elif scenario == "amount":

            amount = profile["normal_amount"] * np.random.uniform(
                5,
                15
            )

            amount = np.clip(
                amount,
                5000,
                100000
            )


        # ----------------------------------------------------
        # Scenario 3: New device
        # ----------------------------------------------------

        elif scenario == "new_device":

            device = np.random.choice(
                devices
            )

            device_age_days = np.random.randint(
                1,
                7
            )

            failed_attempts = np.random.randint(
                1,
                5
            )


        # ----------------------------------------------------
        # Scenario 4: Location anomaly
        # ----------------------------------------------------

        elif scenario == "location":

            other_locations = [
                x for x in locations
                if x != profile["home_location"]
            ]

            location = np.random.choice(
                other_locations
            )

            location_change = 1

            amount *= np.random.uniform(
                2,
                5
            )


        # ----------------------------------------------------
        # Scenario 5: Credential abuse pattern
        # ----------------------------------------------------

        elif scenario == "credential_abuse":

            failed_attempts = np.random.randint(
                3,
                8
            )

            transactions_last_10min = np.random.randint(
                3,
                9
            )

            device = np.random.choice(
                devices
            )

            device_age_days = np.random.randint(
                1,
                30
            )


        # ----------------------------------------------------
        # Scenario 6: Combined attack pattern
        # ----------------------------------------------------

        elif scenario == "combined":

            amount = profile["normal_amount"] * np.random.uniform(
                4,
                12
            )

            amount = np.clip(
                amount,
                5000,
                100000
            )

            transactions_last_10min = np.random.randint(
                4,
                12
            )

            failed_attempts = np.random.randint(
                2,
                7
            )

            device = np.random.choice(
                devices
            )

            device_age_days = np.random.randint(
                1,
                14
            )

            other_locations = [
                x for x in locations
                if x != profile["home_location"]
            ]

            location = np.random.choice(
                other_locations
            )

            location_change = 1


    # --------------------------------------------------------
    # Add realistic noise
    # --------------------------------------------------------

    # Some legitimate users can have unusual behavior
    if not is_fraud:

        if np.random.random() < 0.04:

            transactions_last_10min = np.random.randint(
                3,
                7
            )

        if np.random.random() < 0.03:

            amount *= np.random.uniform(
                2.5,
                5
            )

        if np.random.random() < 0.03:

            device_age_days = np.random.randint(
                1,
                20
            )

        if np.random.random() < 0.02:

            location_change = 1


    # --------------------------------------------------------
    # Final amount
    # --------------------------------------------------------

    amount = round(
        float(
            np.clip(
                amount,
                50,
                100000
            )
        ),
        2
    )


    # --------------------------------------------------------
    # Behavioral features
    # --------------------------------------------------------

    amount_deviation = (
        amount /
        max(
            profile["normal_amount"],
            1
        )
    )


    # --------------------------------------------------------
    # Combined behavioral signal
    # --------------------------------------------------------

    behavior_risk_count = 0

    if transactions_last_10min >= 4:
        behavior_risk_count += 1

    if failed_attempts >= 3:
        behavior_risk_count += 1

    if device_age_days < 14:
        behavior_risk_count += 1

    if location_change == 1:
        behavior_risk_count += 1

    if amount_deviation > 3:
        behavior_risk_count += 1


    rows.append({

        "transaction_id":
            f"TX_{i:07d}",

        "customer_id":
            customer,

        "merchant_id":
            np.random.choice(
                merchants
            ),

        "device_id":
            device,

        "ip_id":
            ip,

        "timestamp":
            timestamp,

        "amount":
            amount,

        "location":
            location,

        "account_age_days":
            profile["account_age_days"],

        "device_age_days":
            device_age_days,

        "transactions_last_10min":
            transactions_last_10min,

        "failed_attempts":
            failed_attempts,

        "location_change":
            location_change,

        "amount_deviation":
            round(
                amount_deviation,
                3
            ),

        "behavior_risk_count":
            behavior_risk_count,

        "is_fraud":
            is_fraud
    })


# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(rows)

df = df.sort_values(
    "timestamp"
).reset_index(drop=True)


# ============================================================
# SAVE
# ============================================================

df.to_csv(
    "transactions_v2.csv",
    index=False
)


print("=" * 65)
print("RISKGRAPH AI — DATASET V2")
print("=" * 65)

print(
    f"Transactions : {len(df):,}"
)

print(
    f"Fraud cases  : {df['is_fraud'].sum():,}"
)

print(
    f"Fraud rate   : "
    f"{df['is_fraud'].mean() * 100:.2f}%"
)

print("=" * 65)

print("\nFraud distribution:")
print(
    df["is_fraud"].value_counts()
)

print("\nPreview:")
display(df.head())

RISKGRAPH AI — DATASET V2
Transactions : 20,000
Fraud cases  : 1,225
Fraud rate   : 6.12%

Fraud distribution:
is_fraud
0    18775
1     1225
Name: count, dtype: int64

Preview:


,transaction_id,customer_id,merchant_id,device_id,ip_id,timestamp,amount,location,account_age_days,device_age_days,transactions_last_10min,failed_attempts,location_change,amount_deviation,behavior_risk_count,is_fraud
0,TX_0018244,CUST_01791,MER_0044,DEV_01935,IP_00633,2026-01-01 00:05:00,233.55,Kolkata,2152,1029,1,2,0,1.081,0,0
1,TX_0016137,CUST_03024,MER_0065,DEV_02313,IP_04623,2026-01-01 00:16:00,2427.66,Hyderabad,667,1034,1,0,0,1.094,0,0
2,TX_0019320,CUST_01982,MER_0064,DEV_04622,IP_04132,2026-01-01 00:28:00,1347.98,Kolkata,1101,609,0,0,0,1.758,0,0
3,TX_0019385,CUST_03754,MER_0242,DEV_02950,IP_04540,2026-01-01 00:54:00,682.27,Chennai,445,393,0,0,0,1.162,0,0
4,TX_0005847,CUST_03535,MER_0113,DEV_01316,IP_01719,2026-01-01 01:05:00,606.03,Bengaluru,810,306,3,0,0,0.813,0,0


In [20]:
#19
print(
    df.groupby("is_fraud")[
        [
            "amount_deviation",
            "transactions_last_10min",
            "failed_attempts",
            "device_age_days",
            "location_change",
            "behavior_risk_count"
        ]
    ].mean()
)

          amount_deviation  transactions_last_10min  failed_attempts  \
is_fraud                                                               
0                 1.166788                 1.244900         0.402024   
1                 4.642475                 4.909388         2.710204   

          device_age_days  location_change  behavior_risk_count  
is_fraud                                                         
0               595.54269         0.021092             0.129374  
1               282.14449         0.374694             2.419592  


In [22]:
#20
# ============================================================
# RISKGRAPH AI — PHASE 2 V2
# FEATURE ENGINEERING
# ============================================================

model_df = df.copy()

model_df["timestamp"] = pd.to_datetime(
    model_df["timestamp"]
)

# Time features
model_df["hour"] = (
    model_df["timestamp"].dt.hour
)

model_df["day_of_week"] = (
    model_df["timestamp"].dt.dayofweek
)

model_df["is_weekend"] = (
    model_df["day_of_week"] >= 5
).astype(int)


# Amount features
model_df["log_amount"] = np.log1p(
    model_df["amount"]
)

model_df["high_value_transaction"] = (
    model_df["amount"] > 10000
).astype(int)


# Behavioral indicators
model_df["high_velocity"] = (
    model_df["transactions_last_10min"] >= 4
).astype(int)

model_df["high_failure_activity"] = (
    model_df["failed_attempts"] >= 3
).astype(int)

model_df["new_device"] = (
    model_df["device_age_days"] < 14
).astype(int)

model_df["new_account"] = (
    model_df["account_age_days"] < 60
).astype(int)


features_v2 = [

    "amount",

    "log_amount",

    "account_age_days",

    "device_age_days",

    "transactions_last_10min",

    "failed_attempts",

    "location_change",

    "amount_deviation",

    "behavior_risk_count",

    "hour",

    "day_of_week",

    "is_weekend",

    "high_value_transaction",

    "high_velocity",

    "high_failure_activity",

    "new_device",

    "new_account"
]

X_v2 = model_df[features_v2]

y_v2 = model_df["is_fraud"]


print("Feature matrix:", X_v2.shape)
print("Target:", y_v2.shape)

Feature matrix: (20000, 17)
Target: (20000,)


In [23]:
#21
from sklearn.model_selection import train_test_split

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(

    X_v2,

    y_v2,

    test_size=0.20,

    random_state=42,

    stratify=y_v2
)

print(
    "Training samples:",
    len(X_train_v2)
)

print(
    "Held-out test samples:",
    len(X_test_v2)
)

print(
    "\nTraining fraud rate:",
    f"{y_train_v2.mean() * 100:.2f}%"
)

print(
    "Test fraud rate:",
    f"{y_test_v2.mean() * 100:.2f}%"
)

Training samples: 16000
Held-out test samples: 4000

Training fraud rate: 6.12%
Test fraud rate: 6.12%


In [24]:
#22
from sklearn.ensemble import RandomForestClassifier

model_v2 = RandomForestClassifier(

    n_estimators=300,

    max_depth=10,

    min_samples_leaf=5,

    class_weight="balanced",

    random_state=42,

    n_jobs=-1
)

print("Training RiskGraph V2 model...")

model_v2.fit(
    X_train_v2,
    y_train_v2
)

print("Training complete.")

Training RiskGraph V2 model...
Training complete.


In [25]:
#23
y_probability_v2 = model_v2.predict_proba(
    X_test_v2
)[:, 1]

threshold_v2 = 0.50

y_pred_v2 = (
    y_probability_v2 >= threshold_v2
).astype(int)

print("Predictions generated.")

print(
    "\nFirst 10 fraud probabilities:"
)

print(
    np.round(
        y_probability_v2[:10],
        4
    )
)

Predictions generated.

First 10 fraud probabilities:
[0.     0.     0.     0.     0.     0.     0.9509 0.     0.     0.    ]


In [26]:
#24
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

precision_v2 = precision_score(
    y_test_v2,
    y_pred_v2,
    zero_division=0
)

recall_v2 = recall_score(
    y_test_v2,
    y_pred_v2,
    zero_division=0
)

f1_v2 = f1_score(
    y_test_v2,
    y_pred_v2,
    zero_division=0
)

roc_auc_v2 = roc_auc_score(
    y_test_v2,
    y_probability_v2
)

cm_v2 = confusion_matrix(
    y_test_v2,
    y_pred_v2
)


print("=" * 65)
print("RISKGRAPH AI — V2 HELD-OUT TEST RESULTS")
print("=" * 65)

print(
    f"\nPrecision : {precision_v2:.4f}"
)

print(
    f"Recall    : {recall_v2:.4f}"
)

print(
    f"F1 Score  : {f1_v2:.4f}"
)

print(
    f"ROC-AUC   : {roc_auc_v2:.4f}"
)

print("\nConfusion Matrix:")
print(cm_v2)

print("\nClassification Report:")

print(
    classification_report(
        y_test_v2,
        y_pred_v2,
        target_names=[
            "Legitimate",
            "Fraud"
        ],
        zero_division=0
    )
)

RISKGRAPH AI — V2 HELD-OUT TEST RESULTS

Precision : 0.8351
Recall    : 0.9714
F1 Score  : 0.8981
ROC-AUC   : 0.9979

Confusion Matrix:
[[3708   47]
 [   7  238]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      0.99      0.99      3755
       Fraud       0.84      0.97      0.90       245

    accuracy                           0.99      4000
   macro avg       0.92      0.98      0.95      4000
weighted avg       0.99      0.99      0.99      4000



In [27]:
#25
importance_v2 = pd.DataFrame({

    "feature": features_v2,

    "importance":
        model_v2.feature_importances_

}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)


print(
    importance_v2.head(15)
)

                    feature  importance
0       behavior_risk_count    0.353380
1           failed_attempts    0.130419
2          amount_deviation    0.085489
3     high_failure_activity    0.084523
4           device_age_days    0.079914
5   transactions_last_10min    0.068857
6                log_amount    0.040008
7             high_velocity    0.039825
8                    amount    0.038304
9                new_device    0.035037
10          location_change    0.027047
11   high_value_transaction    0.007352
12         account_age_days    0.004651
13                     hour    0.003117
14              day_of_week    0.001763


In [28]:
#26
import joblib

joblib.dump(
    {
        "model": model_v2,
        "features": features_v2
    },
    "riskgraph_fraud_model_v2.joblib"
)

print(
    "V2 model saved successfully."
)

V2 model saved successfully.


In [29]:
#27
# ============================================================
# RISKGRAPH AI — TEMPORAL HOLDOUT EVALUATION
# ============================================================

# Sort the complete dataset by transaction time
temporal_df = model_df.sort_values(
    "timestamp"
).reset_index(drop=True)

# Use the first 80% as historical training data
# and the final 20% as future unseen data.

split_index = int(
    len(temporal_df) * 0.80
)

train_temporal = temporal_df.iloc[
    :split_index
]

test_temporal = temporal_df.iloc[
    split_index:
]

X_train_temporal = train_temporal[
    features_v2
]

y_train_temporal = train_temporal[
    "is_fraud"
]

X_test_temporal = test_temporal[
    features_v2
]

y_test_temporal = test_temporal[
    "is_fraud"
]

print("=" * 65)
print("TEMPORAL HOLDOUT")
print("=" * 65)

print(
    "\nTraining transactions:",
    len(train_temporal)
)

print(
    "Future test transactions:",
    len(test_temporal)
)

print(
    "\nTraining fraud rate:",
    f"{y_train_temporal.mean() * 100:.2f}%"
)

print(
    "Future test fraud rate:",
    f"{y_test_temporal.mean() * 100:.2f}%"
)

print(
    "\nTraining period:",
    train_temporal["timestamp"].min(),
    "→",
    train_temporal["timestamp"].max()
)

print(
    "Test period:",
    test_temporal["timestamp"].min(),
    "→",
    test_temporal["timestamp"].max()
)

TEMPORAL HOLDOUT

Training transactions: 16000
Future test transactions: 4000

Training fraud rate: 6.13%
Future test fraud rate: 6.10%

Training period: 2026-01-01 00:05:00 → 2026-05-24 10:59:00
Test period: 2026-05-24 11:16:00 → 2026-06-29 23:53:00


In [30]:
#28
# ============================================================
# TRAIN TEMPORAL MODEL
# ============================================================

temporal_model = RandomForestClassifier(

    n_estimators=300,

    max_depth=10,

    min_samples_leaf=5,

    class_weight="balanced",

    random_state=42,

    n_jobs=-1
)

print("Training temporal model...")

temporal_model.fit(
    X_train_temporal,
    y_train_temporal
)

print("Temporal model trained.")

Training temporal model...
Temporal model trained.


In [31]:
#29
# ============================================================
# FUTURE TEST PREDICTIONS
# ============================================================

temporal_probability = (
    temporal_model.predict_proba(
        X_test_temporal
    )[:, 1]
)

temporal_threshold = 0.50

temporal_predictions = (
    temporal_probability >= temporal_threshold
).astype(int)


# Metrics

temporal_precision = precision_score(
    y_test_temporal,
    temporal_predictions,
    zero_division=0
)

temporal_recall = recall_score(
    y_test_temporal,
    temporal_predictions,
    zero_division=0
)

temporal_f1 = f1_score(
    y_test_temporal,
    temporal_predictions,
    zero_division=0
)

temporal_auc = roc_auc_score(
    y_test_temporal,
    temporal_probability
)

temporal_cm = confusion_matrix(
    y_test_temporal,
    temporal_predictions
)


print("=" * 65)
print("RISKGRAPH AI — FUTURE HOLDOUT RESULTS")
print("=" * 65)

print(
    f"\nPrecision : {temporal_precision:.4f}"
)

print(
    f"Recall    : {temporal_recall:.4f}"
)

print(
    f"F1 Score  : {temporal_f1:.4f}"
)

print(
    f"ROC-AUC   : {temporal_auc:.4f}"
)

print("\nConfusion Matrix:")

print(
    temporal_cm
)

RISKGRAPH AI — FUTURE HOLDOUT RESULTS

Precision : 0.8511
Recall    : 0.9836
F1 Score  : 0.9125
ROC-AUC   : 0.9985

Confusion Matrix:
[[3714   42]
 [   4  240]]


In [32]:
#30
# ============================================================
# RISKGRAPH AI — PHASE 3
# BEHAVIORAL ANOMALY ENGINE
# ============================================================

from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

# Behavioral features used by anomaly detector
anomaly_features = [
    "amount_deviation",
    "transactions_last_10min",
    "failed_attempts",
    "device_age_days",
    "location_change",
    "account_age_days",
    "behavior_risk_count"
]

# ------------------------------------------------------------
# Train ONLY on legitimate historical transactions
# ------------------------------------------------------------

legitimate_training = train_temporal[
    train_temporal["is_fraud"] == 0
]

X_anomaly_train = legitimate_training[
    anomaly_features
]

X_anomaly_test = test_temporal[
    anomaly_features
]

y_anomaly_test = test_temporal[
    "is_fraud"
]

print("Legitimate training transactions:",
      len(X_anomaly_train))

print("Future test transactions:",
      len(X_anomaly_test))

Legitimate training transactions: 15019
Future test transactions: 4000


In [33]:
#31
# ============================================================
# TRAIN ISOLATION FOREST
# ============================================================

anomaly_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

print("Training behavioral anomaly engine...")

anomaly_model.fit(
    X_anomaly_train
)

print("Anomaly engine trained.")

Training behavioral anomaly engine...
Anomaly engine trained.


In [34]:
#32
# ============================================================
# GENERATE ANOMALY SCORES
# ============================================================

# Isolation Forest:
# Higher decision_function = more normal
# Lower decision_function = more anomalous

raw_anomaly_score = -anomaly_model.decision_function(
    X_anomaly_test
)

# Convert to 0-100 using percentile ranking
# based on the legitimate training distribution.

training_raw_scores = -anomaly_model.decision_function(
    X_anomaly_train
)

sorted_training_scores = np.sort(
    training_raw_scores
)

anomaly_percentiles = (
    np.searchsorted(
        sorted_training_scores,
        raw_anomaly_score,
        side="right"
    )
    / len(sorted_training_scores)
)

anomaly_score = (
    anomaly_percentiles * 100
)

anomaly_score = np.clip(
    anomaly_score,
    0,
    100
)

print("Anomaly scores generated.")

print(
    "\nFirst 10 anomaly scores:"
)

print(
    np.round(
        anomaly_score[:10],
        2
    )
)

Anomaly scores generated.

First 10 anomaly scores:
[ 2.23 53.61 52.8  59.5  59.03 16.69 20.41  3.87 19.13 60.46]


In [35]:
#33
# ============================================================
# ANOMALY DETECTOR EVALUATION
# ============================================================

anomaly_auc = roc_auc_score(
    y_anomaly_test,
    anomaly_score
)

print("=" * 65)
print("RISKGRAPH AI — ANOMALY ENGINE EVALUATION")
print("=" * 65)

print(
    f"\nAnomaly ROC-AUC: {anomaly_auc:.4f}"
)

print("\nAverage anomaly score:")

print(
    pd.DataFrame({
        "is_fraud": y_anomaly_test.values,
        "anomaly_score": anomaly_score
    })
    .groupby("is_fraud")
    .mean()
)

RISKGRAPH AI — ANOMALY ENGINE EVALUATION

Anomaly ROC-AUC: 0.9912

Average anomaly score:
          anomaly_score
is_fraud               
0             49.574584
1             98.848480


In [36]:
#34
# ============================================================
# BUILD FUTURE TEST INVESTIGATION TABLE
# ============================================================

investigation_df = test_temporal[
    [
        "transaction_id",
        "customer_id",
        "merchant_id",
        "device_id",
        "ip_id",
        "timestamp",
        "amount",
        "location",
        "account_age_days",
        "device_age_days",
        "transactions_last_10min",
        "failed_attempts",
        "location_change",
        "amount_deviation",
        "behavior_risk_count",
        "is_fraud"
    ]
].copy()

investigation_df["fraud_probability"] = (
    temporal_probability
)

investigation_df["anomaly_score"] = (
    anomaly_score
)

investigation_df["model_decision"] = np.where(
    temporal_probability >= 0.50,
    "REVIEW",
    "APPROVE"
)

investigation_df.head()


,transaction_id,customer_id,merchant_id,device_id,ip_id,timestamp,amount,location,account_age_days,device_age_days,transactions_last_10min,failed_attempts,location_change,amount_deviation,behavior_risk_count,is_fraud,fraud_probability,anomaly_score,model_decision
16000,TX_0013555,CUST_01231,MER_0268,DEV_03767,IP_03770,2026-05-24 11:16:00,1736.65,Chennai,836,357,1,0,0,0.780,0,0,0.000055,2.230508,APPROVE
16001,TX_0010096,CUST_00508,MER_0065,DEV_03489,IP_04858,2026-05-24 11:18:00,3739.89,Mumbai,1386,631,1,0,0,1.875,0,0,0.000000,53.612091,APPROVE
16002,TX_0008297,CUST_00225,MER_0231,DEV_00715,IP_05383,2026-05-24 11:33:00,3348.69,Jaipur,1786,563,1,0,0,1.304,0,0,0.000000,52.799787,APPROVE
16003,TX_0007102,CUST_03419,MER_0021,DEV_00525,IP_05775,2026-05-24 11:43:00,597.14,Ahmedabad,1189,159,2,1,0,0.709,0,0,0.000000,59.497969,APPROVE
16004,TX_0015608,CUST_03025,MER_0229,DEV_01596,IP_03362,2026-05-24 11:50:00,433.68,Chennai,1067,919,2,1,0,0.480,0,0,0.000000,59.031893,APPROVE


In [37]:
#35
# ============================================================
# RISKGRAPH AI — PHASE 4
# ENTITY / GRAPH INTELLIGENCE
# ============================================================

from collections import defaultdict

print("=" * 65)
print("RISKGRAPH AI — ENTITY GRAPH ENGINE")
print("=" * 65)


# ------------------------------------------------------------
# Historical training data only
# ------------------------------------------------------------

historical = train_temporal.copy()

future = test_temporal.copy()


# ------------------------------------------------------------
# CUSTOMER COUNTS PER DEVICE
# ------------------------------------------------------------

device_customers = (
    historical
    .groupby("device_id")["customer_id"]
    .nunique()
)


# ------------------------------------------------------------
# CUSTOMER COUNTS PER IP
# ------------------------------------------------------------

ip_customers = (
    historical
    .groupby("ip_id")["customer_id"]
    .nunique()
)


# ------------------------------------------------------------
# TRANSACTION COUNTS PER DEVICE
# ------------------------------------------------------------

device_transactions = (
    historical
    .groupby("device_id")
    .size()
)


# ------------------------------------------------------------
# TRANSACTION COUNTS PER IP
# ------------------------------------------------------------

ip_transactions = (
    historical
    .groupby("ip_id")
    .size()
)


print(
    "Unique devices:",
    historical["device_id"].nunique()
)

print(
    "Unique IPs:",
    historical["ip_id"].nunique()
)

print(
    "Devices shared by multiple customers:",
    (device_customers > 1).sum()
)

print(
    "IPs shared by multiple customers:",
    (ip_customers > 1).sum()
)

RISKGRAPH AI — ENTITY GRAPH ENGINE
Unique devices: 2955
Unique IPs: 2860
Devices shared by multiple customers: 1111
IPs shared by multiple customers: 845


In [38]:
#36
# ============================================================
# CREATE GRAPH FEATURES FOR FUTURE TRANSACTIONS
# ============================================================

graph_features = future[
    [
        "transaction_id",
        "customer_id",
        "device_id",
        "ip_id"
    ]
].copy()


# Device relationship features

graph_features["device_customer_count"] = (
    graph_features["device_id"]
    .map(device_customers)
    .fillna(0)
)

graph_features["device_transaction_count"] = (
    graph_features["device_id"]
    .map(device_transactions)
    .fillna(0)
)


# IP relationship features

graph_features["ip_customer_count"] = (
    graph_features["ip_id"]
    .map(ip_customers)
    .fillna(0)
)

graph_features["ip_transaction_count"] = (
    graph_features["ip_id"]
    .map(ip_transactions)
    .fillna(0)
)


# ------------------------------------------------------------
# Entity sharing indicators
# ------------------------------------------------------------

graph_features["shared_device"] = (
    graph_features["device_customer_count"] > 1
).astype(int)

graph_features["shared_ip"] = (
    graph_features["ip_customer_count"] > 1
).astype(int)


# ------------------------------------------------------------
# Combined entity risk
# ------------------------------------------------------------

graph_features["entity_risk_count"] = (
    graph_features["shared_device"]
    + graph_features["shared_ip"]
    + (
        graph_features["device_transaction_count"] >= 10
    ).astype(int)
    + (
        graph_features["ip_transaction_count"] >= 10
    ).astype(int)
)


graph_features.head()

,transaction_id,customer_id,device_id,ip_id,device_customer_count,device_transaction_count,ip_customer_count,ip_transaction_count,shared_device,shared_ip,entity_risk_count
16000,TX_0013555,CUST_01231,DEV_03767,IP_03770,2.0,7.0,1.0,3.0,1,0,1
16001,TX_0010096,CUST_00508,DEV_03489,IP_04858,3.0,4.0,2.0,4.0,1,1,2
16002,TX_0008297,CUST_00225,DEV_00715,IP_05383,2.0,6.0,1.0,2.0,1,0,1
16003,TX_0007102,CUST_03419,DEV_00525,IP_05775,1.0,3.0,3.0,14.0,0,1,2
16004,TX_0015608,CUST_03025,DEV_01596,IP_03362,2.0,6.0,2.0,10.0,1,1,3


In [39]:
#37
# ============================================================
# ENTITY GRAPH RISK SCORE
# ============================================================

graph_features["graph_risk_score"] = (

    np.minimum(
        graph_features["device_customer_count"] * 8,
        30
    )

    +

    np.minimum(
        graph_features["ip_customer_count"] * 8,
        30
    )

    +

    np.minimum(
        graph_features["device_transaction_count"] * 1.5,
        20
    )

    +

    np.minimum(
        graph_features["ip_transaction_count"] * 1.5,
        20
    )
)


graph_features["graph_risk_score"] = np.clip(
    graph_features["graph_risk_score"],
    0,
    100
)


print(
    graph_features[
        [
            "transaction_id",
            "device_customer_count",
            "ip_customer_count",
            "device_transaction_count",
            "ip_transaction_count",
            "graph_risk_score"
        ]
    ].head(20)
)

      transaction_id  device_customer_count  ip_customer_count  \
16000     TX_0013555                    2.0                1.0   
16001     TX_0010096                    3.0                2.0   
16002     TX_0008297                    2.0                1.0   
16003     TX_0007102                    1.0                3.0   
16004     TX_0015608                    2.0                2.0   
16005     TX_0010123                    2.0                3.0   
16006     TX_0017168                    2.0                1.0   
16007     TX_0010503                    2.0                1.0   
16008     TX_0007143                    3.0                1.0   
16009     TX_0004738                    1.0                2.0   
16010     TX_0001867                    2.0                1.0   
16011     TX_0001190                    1.0                2.0   
16012     TX_0013175                    1.0                2.0   
16013     TX_0007223                    1.0                2.0   
16014     

In [40]:
#38
# ============================================================
# MERGE GRAPH SIGNALS INTO INVESTIGATION TABLE
# ============================================================

graph_columns = [
    "transaction_id",
    "device_customer_count",
    "ip_customer_count",
    "device_transaction_count",
    "ip_transaction_count",
    "shared_device",
    "shared_ip",
    "entity_risk_count",
    "graph_risk_score"
]

investigation_df = investigation_df.merge(
    graph_features[graph_columns],
    on="transaction_id",
    how="left"
)


print(
    investigation_df[
        [
            "transaction_id",
            "fraud_probability",
            "anomaly_score",
            "graph_risk_score"
        ]
    ].head(10)
)

  transaction_id  fraud_probability  anomaly_score  graph_risk_score
0     TX_0013555           0.000055       2.230508              39.0
1     TX_0010096           0.000000      53.612091              52.0
2     TX_0008297           0.000000      52.799787              36.0
3     TX_0007102           0.000000      59.497969              56.5
4     TX_0015608           0.000000      59.031893              56.0
5     TX_0010123           0.000055      16.692190              70.5
6     TX_0017168           0.000000      20.407484              45.0
7     TX_0010503           0.000092       3.868433              40.5
8     TX_0007143           0.000037      19.129103              47.0
9     TX_0004738           0.000037      60.463413              51.0


In [41]:
#39
# ============================================================
# GRAPH SIGNAL EVALUATION
# ============================================================

graph_evaluation = (
    investigation_df
    .groupby("is_fraud")[
        [
            "device_customer_count",
            "ip_customer_count",
            "device_transaction_count",
            "ip_transaction_count",
            "graph_risk_score"
        ]
    ]
    .mean()
)


print("=" * 65)
print("GRAPH SIGNAL EVALUATION")
print("=" * 65)

print(
    graph_evaluation
)

GRAPH SIGNAL EVALUATION
          device_customer_count  ip_customer_count  device_transaction_count  \
is_fraud                                                                       
0                      1.863419           1.671725                  7.012247   
1                      1.266393           1.614754                  4.815574   

          ip_transaction_count  graph_risk_score  
is_fraud                                          
0                     6.752396         47.873136  
1                     6.889344         39.918033  


In [43]:
graph_auc = roc_auc_score(
    investigation_df["is_fraud"],
    investigation_df["graph_risk_score"]
)

print(
    f"Graph Risk ROC-AUC: {graph_auc:.4f}"
)

Graph Risk ROC-AUC: 0.3673


In [44]:
#40
# ============================================================
# RISKGRAPH AI — PHASE 5
# RISK FUSION ENGINE
# ============================================================

risk_df = investigation_df.copy()

# ------------------------------------------------------------
# 1. Normalize fraud probability
# ------------------------------------------------------------

risk_df["fraud_signal"] = (
    risk_df["fraud_probability"] * 100
)


# ------------------------------------------------------------
# 2. Normalize anomaly signal
# ------------------------------------------------------------

risk_df["anomaly_signal"] = (
    risk_df["anomaly_score"]
)


# ------------------------------------------------------------
# 3. Risk fusion
#
# Fraud probability gets more weight because it is our
# supervised predictive model.
#
# Anomaly detection provides an independent behavioral signal.
# ------------------------------------------------------------

risk_df["raw_risk_score"] = (

    0.70 * risk_df["fraud_signal"]

    +

    0.30 * risk_df["anomaly_signal"]
)


risk_df["raw_risk_score"] = np.clip(
    risk_df["raw_risk_score"],
    0,
    100
)


print("=" * 65)
print("RISKGRAPH AI — RISK FUSION")
print("=" * 65)

display(
    risk_df[
        [
            "transaction_id",
            "amount",
            "fraud_probability",
            "anomaly_score",
            "raw_risk_score",
            "is_fraud"
        ]
    ].head(10)
)

RISKGRAPH AI — RISK FUSION


,transaction_id,amount,fraud_probability,anomaly_score,raw_risk_score,is_fraud
0,TX_0013555,1736.65,0.000055,2.230508,0.673009,0
1,TX_0010096,3739.89,0.000000,53.612091,16.083627,0
2,TX_0008297,3348.69,0.000000,52.799787,15.839936,0
3,TX_0007102,597.14,0.000000,59.497969,17.849391,0
4,TX_0015608,433.68,0.000000,59.031893,17.709568,0
5,TX_0010123,1747.10,0.000055,16.692190,5.011513,0
6,TX_0017168,1041.26,0.000000,20.407484,6.122245,0
7,TX_0010503,1500.92,0.000092,3.868433,1.166945,0
8,TX_0007143,504.18,0.000037,19.129103,5.741289,0
9,TX_0004738,899.51,0.000037,60.463413,18.141582,0


In [45]:
#41
# ============================================================
# EXPECTED FRAUD LOSS
# ============================================================

risk_df["expected_fraud_loss"] = (

    risk_df["fraud_probability"]
    *
    risk_df["amount"]
)


print(
    risk_df[
        [
            "transaction_id",
            "amount",
            "fraud_probability",
            "expected_fraud_loss"
        ]
    ].head(10)
)

  transaction_id   amount  fraud_probability  expected_fraud_loss
0     TX_0013555  1736.65           0.000055             0.095677
1     TX_0010096  3739.89           0.000000             0.000000
2     TX_0008297  3348.69           0.000000             0.000000
3     TX_0007102   597.14           0.000000             0.000000
4     TX_0015608   433.68           0.000000             0.000000
5     TX_0010123  1747.10           0.000055             0.096253
6     TX_0017168  1041.26           0.000000             0.000000
7     TX_0010503  1500.92           0.000092             0.137546
8     TX_0007143   504.18           0.000037             0.018427
9     TX_0004738   899.51           0.000037             0.032876


In [46]:
#42
# ============================================================
# COST MODEL
# ============================================================

# These are modelling assumptions for the prototype.
# They are NOT Razorpay's actual operational costs.

VERIFY_COST = 25.0
REVIEW_COST = 75.0

# Estimated percentage of transaction value lost
# when a legitimate transaction is incorrectly stopped/reviewed.
FALSE_POSITIVE_RATE = 0.01


risk_df["false_positive_cost"] = (
    risk_df["amount"]
    * FALSE_POSITIVE_RATE
)


print("Cost assumptions:")
print(f"Verification cost : ₹{VERIFY_COST}")
print(f"Review cost       : ₹{REVIEW_COST}")
print(
    f"False-positive rate assumption : "
    f"{FALSE_POSITIVE_RATE * 100:.1f}%"
)

Cost assumptions:
Verification cost : ₹25.0
Review cost       : ₹75.0
False-positive rate assumption : 1.0%


In [47]:
#43
# ============================================================
# ACTION COST MODEL
# ============================================================

VERIFY_FRAUD_REDUCTION = 0.80
REVIEW_FRAUD_REDUCTION = 0.95


# ------------------------------------------------------------
# APPROVE
# ------------------------------------------------------------

risk_df["approve_cost"] = (
    risk_df["expected_fraud_loss"]
)


# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

risk_df["verify_cost"] = (

    VERIFY_COST

    +

    risk_df["expected_fraud_loss"]
    *
    (1 - VERIFY_FRAUD_REDUCTION)
)


# ------------------------------------------------------------
# REVIEW
# ------------------------------------------------------------

risk_df["review_cost"] = (

    REVIEW_COST

    +

    risk_df["expected_fraud_loss"]
    *
    (1 - REVIEW_FRAUD_REDUCTION)
)


display(
    risk_df[
        [
            "transaction_id",
            "expected_fraud_loss",
            "approve_cost",
            "verify_cost",
            "review_cost"
        ]
    ].head(10)
)

,transaction_id,expected_fraud_loss,approve_cost,verify_cost,review_cost
0,TX_0013555,0.095677,0.095677,25.019135,75.004784
1,TX_0010096,0.000000,0.000000,25.000000,75.000000
2,TX_0008297,0.000000,0.000000,25.000000,75.000000
3,TX_0007102,0.000000,0.000000,25.000000,75.000000
4,TX_0015608,0.000000,0.000000,25.000000,75.000000
5,TX_0010123,0.096253,0.096253,25.019251,75.004813
6,TX_0017168,0.000000,0.000000,25.000000,75.000000
7,TX_0010503,0.137546,0.137546,25.027509,75.006877
8,TX_0007143,0.018427,0.018427,25.003685,75.000921
9,TX_0004738,0.032876,0.032876,25.006575,75.001644


In [48]:
#44
# ============================================================
# COST-AWARE DECISION ENGINE
# ============================================================

cost_columns = [
    "approve_cost",
    "verify_cost",
    "review_cost"
]

action_map = {
    "approve_cost": "APPROVE",
    "verify_cost": "VERIFY",
    "review_cost": "REVIEW"
}


risk_df["recommended_action"] = (
    risk_df[cost_columns]
    .idxmin(axis=1)
    .map(action_map)
)


risk_df["minimum_expected_cost"] = (
    risk_df[cost_columns]
    .min(axis=1)
)


print("=" * 65)
print("RISKGRAPH AI — COST-AWARE DECISION ENGINE")
print("=" * 65)

display(
    risk_df[
        [
            "transaction_id",
            "amount",
            "fraud_probability",
            "anomaly_score",
            "raw_risk_score",
            "expected_fraud_loss",
            "approve_cost",
            "verify_cost",
            "review_cost",
            "recommended_action"
        ]
    ].head(20)
)

RISKGRAPH AI — COST-AWARE DECISION ENGINE


,transaction_id,amount,fraud_probability,anomaly_score,raw_risk_score,expected_fraud_loss,approve_cost,verify_cost,review_cost,recommended_action
0,TX_0013555,1736.65,0.000055,2.230508,0.673009,0.095677,0.095677,25.019135,75.004784,APPROVE
1,TX_0010096,3739.89,0.000000,53.612091,16.083627,0.000000,0.000000,25.000000,75.000000,APPROVE
2,TX_0008297,3348.69,0.000000,52.799787,15.839936,0.000000,0.000000,25.000000,75.000000,APPROVE
3,TX_0007102,597.14,0.000000,59.497969,17.849391,0.000000,0.000000,25.000000,75.000000,APPROVE
4,TX_0015608,433.68,0.000000,59.031893,17.709568,0.000000,0.000000,25.000000,75.000000,APPROVE
5,TX_0010123,1747.10,0.000055,16.692190,5.011513,0.096253,0.096253,25.019251,75.004813,APPROVE
6,TX_0017168,1041.26,0.000000,20.407484,6.122245,0.000000,0.000000,25.000000,75.000000,APPROVE
7,TX_0010503,1500.92,0.000092,3.868433,1.166945,0.137546,0.137546,25.027509,75.006877,APPROVE
8,TX_0007143,504.18,0.000037,19.129103,5.741289,0.018427,0.018427,25.003685,75.000921,APPROVE
9,TX_0004738,899.51,0.000037,60.463413,18.141582,0.032876,0.032876,25.006575,75.001644,APPROVE


In [49]:
#45
# ============================================================
# DECISION DISTRIBUTION
# ============================================================

decision_distribution = (
    risk_df["recommended_action"]
    .value_counts()
)

print(
    decision_distribution
)

print("\nPercentages:")

print(
    (
        risk_df["recommended_action"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

recommended_action
APPROVE    3305
REVIEW      396
VERIFY      299
Name: count, dtype: int64

Percentages:
recommended_action
APPROVE    82.62
REVIEW      9.90
VERIFY      7.48
Name: proportion, dtype: float64


In [50]:
#46
# ============================================================
# DECISION QUALITY
# ============================================================

risk_df["decision_is_intervention"] = (
    risk_df["recommended_action"]
    != "APPROVE"
).astype(int)

risk_df["actual_fraud"] = (
    risk_df["is_fraud"]
)


decision_cm = confusion_matrix(
    risk_df["actual_fraud"],
    risk_df["decision_is_intervention"]
)


decision_precision = precision_score(
    risk_df["actual_fraud"],
    risk_df["decision_is_intervention"],
    zero_division=0
)


decision_recall = recall_score(
    risk_df["actual_fraud"],
    risk_df["decision_is_intervention"],
    zero_division=0
)


print("=" * 65)
print("RISK DECISION EVALUATION")
print("=" * 65)

print(
    f"\nIntervention Precision: "
    f"{decision_precision:.4f}"
)

print(
    f"Intervention Recall: "
    f"{decision_recall:.4f}"
)

print("\nConfusion Matrix:")

print(
    decision_cm
)

RISK DECISION EVALUATION

Intervention Precision: 0.3511
Intervention Recall: 1.0000

Confusion Matrix:
[[3305  451]
 [   0  244]]


In [51]:
#47
# ============================================================
# FALSE-POSITIVE COST
# ============================================================

false_positive_mask = (
    (risk_df["actual_fraud"] == 0)
    &
    (risk_df["decision_is_intervention"] == 1)
)


false_positive_count = (
    false_positive_mask.sum()
)


false_positive_cost_total = (
    risk_df.loc[
        false_positive_mask,
        "false_positive_cost"
    ].sum()
)


false_positive_amount = (
    risk_df.loc[
        false_positive_mask,
        "amount"
    ].sum()
)


print("=" * 65)
print("FALSE-POSITIVE COST ANALYSIS")
print("=" * 65)

print(
    f"\nFalse positives: "
    f"{false_positive_count:,}"
)

print(
    f"Transaction value involved: "
    f"₹{false_positive_amount:,.2f}"
)

print(
    f"Estimated false-positive cost: "
    f"₹{false_positive_cost_total:,.2f}"
)

FALSE-POSITIVE COST ANALYSIS

False positives: 451
Transaction value involved: ₹1,432,922.86
Estimated false-positive cost: ₹14,329.23


In [52]:
#48
# ============================================================
# FRAUD LOSS AVOIDANCE
# ============================================================

# Baseline:
# Everything gets approved.

baseline_expected_loss = (
    risk_df["expected_fraud_loss"]
    .sum()
)


# Estimate residual fraud loss after decisions

residual_loss = 0.0

for _, row in risk_df.iterrows():

    if row["recommended_action"] == "APPROVE":

        residual_loss += (
            row["expected_fraud_loss"]
        )

    elif row["recommended_action"] == "VERIFY":

        residual_loss += (
            row["expected_fraud_loss"]
            * (1 - VERIFY_FRAUD_REDUCTION)
        )

    elif row["recommended_action"] == "REVIEW":

        residual_loss += (
            row["expected_fraud_loss"]
            * (1 - REVIEW_FRAUD_REDUCTION)
        )


fraud_loss_avoided = (
    baseline_expected_loss
    -
    residual_loss
)


print("=" * 65)
print("MERCHANT LOSS ANALYSIS")
print("=" * 65)

print(
    f"\nBaseline expected fraud loss: "
    f"₹{baseline_expected_loss:,.2f}"
)

print(
    f"Residual expected fraud loss: "
    f"₹{residual_loss:,.2f}"
)

print(
    f"Estimated loss avoided: "
    f"₹{fraud_loss_avoided:,.2f}"
)

MERCHANT LOSS ANALYSIS

Baseline expected fraud loss: ₹2,060,929.95
Residual expected fraud loss: ₹111,460.20
Estimated loss avoided: ₹1,949,469.75


In [53]:
#49
# ============================================================
# FINAL RISK SCORE
# ============================================================

# Financial exposure normalized using the 95th percentile
# of the test-set expected loss.

loss_cap = risk_df[
    "expected_fraud_loss"
].quantile(0.95)


risk_df["financial_exposure_score"] = (
    risk_df["expected_fraud_loss"]
    / max(loss_cap, 1)
    * 100
)

risk_df["financial_exposure_score"] = np.clip(
    risk_df["financial_exposure_score"],
    0,
    100
)


# Final user-facing risk score
#
# 50% supervised fraud signal
# 30% anomaly signal
# 20% financial exposure

risk_df["risk_score"] = (

    0.50 * risk_df["fraud_signal"]

    +

    0.30 * risk_df["anomaly_signal"]

    +

    0.20 * risk_df["financial_exposure_score"]
)


risk_df["risk_score"] = np.clip(
    risk_df["risk_score"],
    0,
    100
)


print(
    risk_df[
        [
            "transaction_id",
            "amount",
            "fraud_probability",
            "anomaly_score",
            "financial_exposure_score",
            "risk_score",
            "recommended_action"
        ]
    ].head(20)
)

   transaction_id   amount  fraud_probability  anomaly_score  \
0      TX_0013555  1736.65           0.000055       2.230508   
1      TX_0010096  3739.89           0.000000      53.612091   
2      TX_0008297  3348.69           0.000000      52.799787   
3      TX_0007102   597.14           0.000000      59.497969   
4      TX_0015608   433.68           0.000000      59.031893   
5      TX_0010123  1747.10           0.000055      16.692190   
6      TX_0017168  1041.26           0.000000      20.407484   
7      TX_0010503  1500.92           0.000092       3.868433   
8      TX_0007143   504.18           0.000037      19.129103   
9      TX_0004738   899.51           0.000037      60.463413   
10     TX_0001867   814.95           0.000037      44.556895   
11     TX_0001190  1449.29           0.000037      69.958053   
12     TX_0013175  6498.18           0.937821      99.800253   
13     TX_0007223  1115.79           0.000092      32.492177   
14     TX_0007719   403.90           0.0

In [54]:
#50
# ============================================================
# RISK EXPLANATION ENGINE
# ============================================================

def generate_explanation(row):

    reasons = []

    if row["fraud_probability"] >= 0.70:
        reasons.append(
            "high fraud probability"
        )

    if row["anomaly_score"] >= 80:
        reasons.append(
            "highly unusual behavioral pattern"
        )

    if row["amount_deviation"] >= 4:
        reasons.append(
            "transaction amount is significantly "
            "above normal customer behavior"
        )

    if row["transactions_last_10min"] >= 4:
        reasons.append(
            "elevated transaction velocity"
        )

    if row["failed_attempts"] >= 3:
        reasons.append(
            "multiple recent failed attempts"
        )

    if row["device_age_days"] < 14:
        reasons.append(
            "newly observed device"
        )

    if row["location_change"] == 1:
        reasons.append(
            "unusual transaction location"
        )

    if not reasons:
        reasons.append(
            "no major risk indicators detected"
        )

    return "; ".join(reasons)


risk_df["risk_explanation"] = (
    risk_df.apply(
        generate_explanation,
        axis=1
    )
)


display(
    risk_df[
        [
            "transaction_id",
            "risk_score",
            "recommended_action",
            "risk_explanation"
        ]
    ].head(20)
)

,transaction_id,risk_score,recommended_action,risk_explanation
0,TX_0013555,0.672889,APPROVE,no major risk indicators detected
1,TX_0010096,16.083627,APPROVE,no major risk indicators detected
2,TX_0008297,15.839936,APPROVE,no major risk indicators detected
3,TX_0007102,17.849391,APPROVE,no major risk indicators detected
4,TX_0015608,17.709568,APPROVE,no major risk indicators detected
5,TX_0010123,5.011400,APPROVE,no major risk indicators detected
6,TX_0017168,6.122245,APPROVE,no major risk indicators detected
7,TX_0010503,1.166524,APPROVE,no major risk indicators detected
8,TX_0007143,5.740748,APPROVE,no major risk indicators detected
9,TX_0004738,18.141189,APPROVE,no major risk indicators detected


In [55]:
#51
# ============================================================
# PHASE 5 — RISK SCORE CALIBRATION
# ============================================================

print("=" * 65)
print("RISK SCORE DISTRIBUTION")
print("=" * 65)

print("\nRisk score statistics:")

print(
    risk_df["risk_score"].describe()
)

print("\nRisk score by actual class:")

print(
    risk_df.groupby("is_fraud")[
        "risk_score"
    ].agg(
        ["count", "mean", "median", "min", "max"]
    )
)

print("\nFraud probability by actual class:")

print(
    risk_df.groupby("is_fraud")[
        "fraud_probability"
    ].agg(
        ["mean", "median", "min", "max"]
    )
)

RISK SCORE DISTRIBUTION

Risk score statistics:
count    4000.000000
mean       21.250796
std        22.307010
min         0.016135
25%         7.870018
50%        15.973969
75%        24.171882
max       100.000000
Name: risk_score, dtype: float64

Risk score by actual class:
          count       mean     median        min         max
is_fraud                                                    
0          3756  16.613413  15.048505   0.016135   96.539085
1           244  92.636087  96.164659  38.809884  100.000000

Fraud probability by actual class:
              mean    median       min       max
is_fraud                                        
0         0.023825  0.000068  0.000000  0.942007
1         0.934244  0.987605  0.195642  1.000000


In [56]:
#52
# ============================================================
# FIND USEFUL RISK SCORE OPERATING POINTS
# ============================================================

threshold_results = []

for threshold in np.arange(
    5,
    96,
    5
):

    intervene = (
        risk_df["risk_score"]
        >= threshold
    ).astype(int)

    precision = precision_score(
        risk_df["is_fraud"],
        intervene,
        zero_division=0
    )

    recall = recall_score(
        risk_df["is_fraud"],
        intervene,
        zero_division=0
    )

    fp = (
        (risk_df["is_fraud"] == 0)
        &
        (intervene == 1)
    ).sum()

    fn = (
        (risk_df["is_fraud"] == 1)
        &
        (intervene == 0)
    ).sum()

    threshold_results.append({

        "risk_threshold": threshold,

        "precision": precision,

        "recall": recall,

        "false_positives": fp,

        "false_negatives": fn
    })


threshold_table = pd.DataFrame(
    threshold_results
)

display(
    threshold_table.round(4)
)

,risk_threshold,precision,recall,false_positives,false_negatives
0,5,0.0728,1.0000,3107,0
1,10,0.0899,1.0000,2469,0
2,15,0.1146,1.0000,1885,0
3,20,0.1640,1.0000,1244,0
4,25,0.2745,1.0000,645,0
5,30,0.4073,1.0000,355,0
6,35,0.4980,1.0000,246,0
7,40,0.5971,0.9959,164,1
8,45,0.6779,0.9918,115,2
9,50,0.7311,0.9918,89,2


In [57]:
#53
# ============================================================
# FINAL RISK DECISION POLICY
# ============================================================

APPROVE_THRESHOLD = 60
REVIEW_THRESHOLD = 75


def risk_decision(score):

    if score < APPROVE_THRESHOLD:
        return "APPROVE"

    elif score < REVIEW_THRESHOLD:
        return "VERIFY"

    else:
        return "REVIEW"


risk_df["final_action"] = (
    risk_df["risk_score"]
    .apply(risk_decision)
)


print("=" * 65)
print("RISKGRAPH AI — FINAL DECISION POLICY")
print("=" * 65)

print(
    risk_df["final_action"]
    .value_counts()
)

print("\nPercentages:")

print(
    (
        risk_df["final_action"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

RISKGRAPH AI — FINAL DECISION POLICY
final_action
APPROVE    3693
REVIEW      266
VERIFY       41
Name: count, dtype: int64

Percentages:
final_action
APPROVE    92.32
REVIEW      6.65
VERIFY      1.03
Name: proportion, dtype: float64


In [58]:
#54
# ============================================================
# FINAL POLICY EVALUATION
# ============================================================

# Anything that is not APPROVE is considered an intervention.

risk_df["final_intervention"] = (
    risk_df["final_action"] != "APPROVE"
).astype(int)


final_precision = precision_score(
    risk_df["is_fraud"],
    risk_df["final_intervention"],
    zero_division=0
)


final_recall = recall_score(
    risk_df["is_fraud"],
    risk_df["final_intervention"],
    zero_division=0
)


final_f1 = f1_score(
    risk_df["is_fraud"],
    risk_df["final_intervention"],
    zero_division=0
)


final_cm = confusion_matrix(
    risk_df["is_fraud"],
    risk_df["final_intervention"]
)


print("=" * 65)
print("FINAL RISK POLICY EVALUATION")
print("=" * 65)

print(
    f"\nIntervention Precision : "
    f"{final_precision:.4f}"
)

print(
    f"Intervention Recall    : "
    f"{final_recall:.4f}"
)

print(
    f"Intervention F1        : "
    f"{final_f1:.4f}"
)

print("\nConfusion Matrix:")

print(final_cm)

FINAL RISK POLICY EVALUATION

Intervention Precision : 0.7785
Intervention Recall    : 0.9795
Intervention F1        : 0.8675

Confusion Matrix:
[[3688   68]
 [   5  239]]


In [59]:
#55
# ============================================================
# FINAL BUSINESS METRICS
# ============================================================

final_fp_mask = (
    (risk_df["is_fraud"] == 0)
    &
    (risk_df["final_intervention"] == 1)
)


final_fn_mask = (
    (risk_df["is_fraud"] == 1)
    &
    (risk_df["final_intervention"] == 0)
)


final_fp_count = (
    final_fp_mask.sum()
)

final_fn_count = (
    final_fn_mask.sum()
)


final_fp_value = (
    risk_df.loc[
        final_fp_mask,
        "amount"
    ].sum()
)


final_fn_value = (
    risk_df.loc[
        final_fn_mask,
        "amount"
    ].sum()
)


final_fp_cost = (
    risk_df.loc[
        final_fp_mask,
        "false_positive_cost"
    ].sum()
)


print("=" * 65)
print("FINAL BUSINESS METRICS")
print("=" * 65)

print(
    f"\nFalse positives : {final_fp_count}"
)

print(
    f"False negatives : {final_fn_count}"
)

print(
    f"\nFalse-positive transaction value:"
    f" ₹{final_fp_value:,.2f}"
)

print(
    f"Missed-fraud transaction value:"
    f" ₹{final_fn_value:,.2f}"
)

print(
    f"\nEstimated false-positive cost:"
    f" ₹{final_fp_cost:,.2f}"
)

FINAL BUSINESS METRICS

False positives : 68
False negatives : 5

False-positive transaction value: ₹483,752.82
Missed-fraud transaction value: ₹4,292.70

Estimated false-positive cost: ₹4,837.53


In [60]:
#56
# ============================================================
# RISK INVESTIGATION RECORD
# ============================================================

final_columns = [
    "transaction_id",
    "customer_id",
    "merchant_id",
    "device_id",
    "ip_id",
    "timestamp",
    "amount",
    "fraud_probability",
    "anomaly_score",
    "graph_risk_score",
    "risk_score",
    "final_action",
    "risk_explanation",
    "is_fraud"
]


risk_cases = (
    risk_df[final_columns]
    .sort_values(
        "risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    risk_cases.head(20)
)

,transaction_id,customer_id,merchant_id,device_id,ip_id,timestamp,amount,fraud_probability,anomaly_score,graph_risk_score,risk_score,final_action,risk_explanation,is_fraud
0,TX_0009808,CUST_02788,MER_0263,DEV_02998,IP_04275,2026-06-09 02:22:00,5000.00,1.000000,100.0,26.5,100.000000,REVIEW,high fraud probability; highly unusual behavio...,1
1,TX_0013395,CUST_03249,MER_0004,DEV_02178,IP_00299,2026-06-09 10:29:00,5241.69,1.000000,100.0,18.5,100.000000,REVIEW,high fraud probability; highly unusual behavio...,1
2,TX_0004599,CUST_01638,MER_0142,DEV_02742,IP_00590,2026-06-06 01:08:00,22258.43,1.000000,100.0,14.0,100.000000,REVIEW,high fraud probability; highly unusual behavio...,1
3,TX_0003514,CUST_00898,MER_0260,DEV_03852,IP_05565,2026-06-03 19:47:00,8396.61,1.000000,100.0,15.5,100.000000,REVIEW,high fraud probability; highly unusual behavio...,1
4,TX_0000280,CUST_02555,MER_0063,DEV_02013,IP_04925,2026-06-16 02:13:00,8834.49,1.000000,100.0,39.0,100.000000,REVIEW,high fraud probability; highly unusual behavio...,1
5,TX_0001156,CUST_03390,MER_0173,DEV_04535,IP_02012,2026-06-04 08:05:00,26255.21,1.000000,100.0,32.5,100.000000,REVIEW,high fraud probability; highly unusual behavio...,1
6,TX_0000149,CUST_00173,MER_0284,DEV_03203,IP_05158,2026-05-29 15:13:00,7157.41,1.000000,100.0,23.5,100.000000,REVIEW,high fraud probability; highly unusual behavio...,1
7,TX_0012444,CUST_00745,MER_0237,DEV_02713,IP_03243,2026-06-24 09:36:00,5000.00,1.000000,100.0,26.5,100.000000,REVIEW,high fraud probability; highly unusual behavio...,1
8,TX_0015671,CUST_02011,MER_0013,DEV_03390,IP_04257,2026-06-24 06:39:00,14088.84,1.000000,100.0,81.0,100.000000,REVIEW,high fraud probability; highly unusual behavio...,1
9,TX_0019584,CUST_03038,MER_0061,DEV_04114,IP_02758,2026-05-29 11:05:00,5000.00,1.000000,100.0,61.0,100.000000,REVIEW,high fraud probability; highly unusual behavio...,1


In [61]:
from google.colab import files

files.download("riskgraph_fraud_model_v2.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [62]:
demo_transactions = risk_df.copy()

demo_transactions = demo_transactions.sort_values(
    "risk_score",
    ascending=False
)

demo_transactions.head(500).to_csv(
    "sample_transactions.csv",
    index=False
)

from google.colab import files

files.download("sample_transactions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>